# 1.1 · 什么是 MCP，为什么需要它

**What MCP is and why it exists**

> **学完本课你能：** 解释 MCP 所解决的集成问题，说出它的参与方和原语，并将一个客户端连接到你的第一个服务器。

官方阅读材料：[What is MCP?](https://modelcontextprotocol.io/docs/2026-07-28/getting-started/intro) · [Specification overview](https://modelcontextprotocol.io/specification/2026-07-28/index) · [Design principles](https://modelcontextprotocol.io/community/design-principles)

从上到下依次运行单元格（`Shift+Enter`），然后改动代码再重新运行。所有内容都在本地运行：服务器位于当前内核、临时目录或 localhost 上。调用真实大语言模型（LLM）的单元格只有在设置了 `ANTHROPIC_API_KEY` 时才会执行，否则自动跳过。

### 1. 用数字看 N × M 问题
在编写任何使用 MCP 通信的代码之前，先把这个经济学论证具体化。改一改数字，看看差距如何扩大。

In [ ]:
def integrations(apps: int, systems: int) -> tuple[int, int]:
    """Connectors to write without a standard vs. with MCP."""
    return apps * systems, apps + systems


for apps, systems in [(3, 5), (10, 50), (40, 2000)]:
    without, with_mcp = integrations(apps, systems)
    print(f"{apps:>3} apps × {systems:>4} systems -> {without:>6} bespoke connectors | "
          f"{with_mcp:>5} MCP components  ({without / with_mcp:.0f}x less work)")

### 2. 第一个服务器：一个工具、一个资源、一个提示模板
官方 Python SDK（`mcp` 2.x）提供了 `MCPServer`。装饰器把普通的 Python 函数变成三种原语（Primitive）：类型注解变成模型看到的 JSON 模式（Schema），文档字符串变成描述。

In [ ]:
import json

from mcp import Client
from mcp.server import MCPServer

notes = {"mcp": "MCP is USB-C for AI applications.", "lsp": "LSP inspired MCP's design."}

server = MCPServer("notes")


@server.tool()
def search_notes(query: str) -> list[str]:
    """Return the ids of notes whose text contains the query (case-insensitive)."""
    return [nid for nid, text in notes.items() if query.lower() in text.lower()]


@server.resource("notes://{note_id}")
def read_note(note_id: str) -> str:
    """The full text of one note."""
    return notes[note_id]


@server.prompt()
def explain(topic: str) -> str:
    """Ask the model to explain a topic to a beginner."""
    return f"Explain {topic} to a new AI engineer in three sentences."


print("server defined:", server.name)

### 3. 连接客户端，发现服务器提供的内容
`Client(server)` 在进程内建立连接——非常适合学习和测试。客户端（Client）在运行时<em>发现</em>能力（Capability）：客户端一侧没有硬编码任何关于工具的信息。

In [ ]:
async with Client(server) as client:
    print("protocol version:", client.protocol_version)
    tools = (await client.list_tools()).tools
    templates = (await client.list_resource_templates()).resource_templates
    prompts = (await client.list_prompts()).prompts

    for t in tools:
        print("TOOL     ", t.name, "-", t.description)
        print("          input schema:", json.dumps(t.input_schema))
    for r in templates:
        print("RESOURCE ", r.uri_template, "-", r.description)
    for p in prompts:
        print("PROMPT   ", p.name, "- args:", [a.name for a in p.arguments or []])

### 4. 使用全部三种原语
注意三种不同的<em>控制方</em>：模型会调用工具（Tool），应用读取资源（Resource），用户挑选提示模板（Prompt）。

In [ ]:
async with Client(server) as client:
    hit = await client.call_tool("search_notes", {"query": "usb"})      # model-controlled
    print("tools/call      ->", hit.structured_content, "| isError:", hit.is_error)

    doc = await client.read_resource("notes://mcp")                      # application-controlled
    print("resources/read  ->", doc.contents[0].text)

    msg = await client.get_prompt("explain", {"topic": "MCP"})          # user-controlled
    print("prompts/get     ->", msg.messages[0].content.text)

### 5. 一次构建，处处集成
两个完全不同的“宿主（Host）”复用<em>同一个</em>服务器，而且两者都不需要事先知道它有哪些工具：一个只列出能力的仪表盘，以及一个通过阅读描述来挑选工具的小型关键词“智能体”。这就是用代码表达的 N + M 论证。

In [ ]:
async def dashboard_host(srv):
    async with Client(srv) as c:
        names = [t.name for t in (await c.list_tools()).tools]
        return f"dashboard: server '{c.server_info.name}' offers {names}"


async def keyword_agent_host(srv, request: str):
    async with Client(srv) as c:
        tools = (await c.list_tools()).tools
        # a stand-in for an LLM: pick the tool whose description shares the most words with the request
        score = lambda t: len(set(request.lower().split()) & set(t.description.lower().split()))
        best = max(tools, key=score)
        result = await c.call_tool(best.name, {"query": request.split()[-1]})
        return f"agent: chose {best.name!r} -> {result.structured_content}"


print(await dashboard_host(server))
print(await keyword_agent_host(server, "find the notes that contain lsp"))

### 6. 窥探线路：通过 stdio 传输的原始 JSON-RPC
在 SDK 之下，MCP 只不过是 JSON-RPC 消息。这里服务器作为一个真正的子进程运行（即 <strong>stdio</strong> 传输（Transport）），我们通过每行写入一个 JSON 对象来与它对话——客户端一侧完全不用 SDK。每个请求都在 `_meta` 中携带 2026-07-28 的信封。

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "notes_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    server = MCPServer("notes")

    @server.tool()
    def search_notes(query: str) -> list[str]:
        """Return ids of notes containing the query."""
        return [n for n in ["mcp", "lsp"] if query.lower() in n]

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))

proc = subprocess.Popen([sys.executable, str(workdir / "notes_server.py")],
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}


def rpc(id_, method, params):
    message = {"jsonrpc": "2.0", "id": id_, "method": method, "params": params}
    print(">>>", json.dumps(message))
    proc.stdin.write(json.dumps(message) + "\n")
    proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    print("<<<", json.dumps(reply, indent=2)[:700], "\n")
    return reply


rpc(1, "server/discover", {"_meta": META})
rpc(2, "tools/call", {"name": "search_notes", "arguments": {"query": "mc"}, "_meta": META})
proc.stdin.close()
exit_code = proc.wait(timeout=10)   # closing stdin is how a stdio client shuts a server down

读一读这些回复：返回的是相同的 `id`，`resultType` 为 `"complete"`，服务器在 `_meta` 中表明自己的身份（`io.modelcontextprotocol/serverInfo`）。`structuredContent` 和文本 `content` 携带的是同一个答案——一个给程序用，一个给模型用。

### 7. 可选：让真实模型来选择工具
如果设置了 `ANTHROPIC_API_KEY`，Claude 会收到这个 MCP 工具（Tool）的名称、描述和模式（Schema）——与服务器所声明的完全一致——并决定是否调用它。否则，这个单元格会解释将会发生什么。

In [ ]:
import os

if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic

    llm = anthropic.Anthropic()
    async with Client(server) as client:
        tools = [{"name": t.name, "description": t.description, "input_schema": t.input_schema}
                 for t in (await client.list_tools()).tools]
        response = llm.messages.create(
            model="claude-opus-5-5", max_tokens=16000, tools=tools,
            messages=[{"role": "user", "content": "Which of my notes mention LSP?"}])
        for block in response.content:
            if block.type == "tool_use":
                result = await client.call_tool(block.name, block.input)
                print(f"Claude called {block.name}({block.input}) -> {result.structured_content}")
            elif block.type == "text":
                print("Claude:", block.text)
else:
    print("skipped: set ANTHROPIC_API_KEY to watch Claude pick `search_notes` from the MCP tool list.")

### 动手练习
1. 在第 2 节的服务器中添加第二个工具（Tool）`add_note(note_id: str, text: str) -> str`，重新运行第 2–5 节，确认两个宿主都能发现它，<strong>而无需修改它们的任何代码</strong>。
2. 在第 6 节中，从 `tools/call` 请求里删除 `"_meta": META`，看看返回的错误码。然后改为发送 `"io.modelcontextprotocol/protocolVersion": "2020-01-01"`。每个错误分别告诉客户端该怎么做？
3. 利用本课中的表格，判断以下每一项应该是工具、资源（Resource）还是提示模板（Prompt），并说明理由：<em>公司的风格指南</em>、<em>发送一条 Slack 消息</em>、<em>“根据这个 diff 写一份发布说明”</em>、<em>今天的值班表</em>。